# Day 4: Embeddings, LLM Annotation, and Semantic Measurement

This notebook is designed to work without paid API access. The LLM annotation section uses a transparent offline stand-in so the validation workflow can be practiced in class. If you have a local Ollama model or an API key, you can replace that stand-in later.

By the end you should be able to:

1. Treat prompting as codebook design.
2. Compare model labels to human labels.
3. Build local dense document vectors from TF-IDF plus SVD.
4. Use vector similarity for nearest neighbors and simple classifiers.
5. Build and stress-test a semantic axis.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

pd.set_option('display.max_colwidth', 140)

In [ ]:
from pathlib import Path


def find_data_dir():
    candidates = [Path('../data'), Path('materials/data'), Path('data')]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the course data directory.')


DATA_DIR = find_data_dir()
DATA_DIR

## spaCy setup

The notebooks use spaCy for tokenization and preprocessing. If `en_core_web_sm` is installed, the same setup also shows POS tags, lemmas, and dependency parses. If not, the notebooks still run with a blank English tokenizer.

In [ ]:
import spacy

try:
    nlp = spacy.load('en_core_web_sm')
    print('Loaded spaCy model: en_core_web_sm')
except OSError:
    nlp = spacy.blank('en')
    nlp.add_pipe('sentencizer')
    print('Using spacy.blank("en"). Install en_core_web_sm for POS tags, lemmas, and dependency parses.')


def spacy_preprocess(text, remove_stop=True, keep_alpha=True, min_len=2):
    """Tokenize and lightly preprocess text with spaCy."""
    doc = nlp.make_doc(str(text))
    tokens = []
    for token in doc:
        if keep_alpha and not token.is_alpha:
            continue
        if remove_stop and token.is_stop:
            continue
        term = token.text.lower()
        if len(term) >= min_len:
            tokens.append(term)
    return tokens


def spacy_analyzer(text):
    return spacy_preprocess(text)


def spacy_analyzer_with_bigrams(text):
    tokens = spacy_preprocess(text)
    bigrams = [tokens[i] + '_' + tokens[i + 1] for i in range(len(tokens) - 1)]
    return tokens + bigrams


def spacy_token_table(text):
    """Show tokenization, preprocessing flags, and parses when a full model is available."""
    doc = nlp(str(text))
    rows = []
    for token in doc:
        rows.append({
            'text': token.text,
            'lemma': token.lemma_ or '(model needed)',
            'pos': token.pos_ or '(model needed)',
            'dep': token.dep_ or '(model needed)',
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'kept_by_preprocess': token.text.lower() in spacy_preprocess(token.text, remove_stop=True)
        })
    return pd.DataFrame(rows)

## 1. Load a small short-text sentiment sample

We use the local TweetEval sentiment sample for short social-media-style text. To keep the first workflow binary and comparable to the previous notebooks, the core example uses only positive and negative labels.

In [ ]:
tweets = pd.read_csv(DATA_DIR / 'tweet_eval_sentiment_sample.csv')
binary_tweets = tweets[tweets['label_name'].isin(['negative', 'positive'])].copy()
binary_tweets = binary_tweets.rename(columns={'label_name': 'polarity'})

pieces = []
for polarity, group in binary_tweets.groupby('polarity'):
    pieces.append(group.sample(250, random_state=42))

sample = (
    pd.concat(pieces, ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

sample[['text', 'polarity', 'split']].head(), sample['polarity'].value_counts()

## 1a. spaCy inspection

Even for LLM-style annotation, it helps to inspect the text with ordinary NLP tools. This makes negation, punctuation, and token-level cues visible.

In [ ]:
spacy_token_table(sample.loc[0, 'text']).head(30)

## 2. Prompting as codebook design

A real LLM call is just one implementation detail. The measurement workflow is broader: define the concept, record the prompt, apply it consistently, validate it, and inspect disagreements.

In [ ]:
prompt_log = pd.DataFrame([
    {
        'version': 'v1_zero_shot',
        'task': 'Classify a social media post as positive or negative.',
        'allowed_labels': 'positive, negative',
        'notes': 'No examples. Offline classroom stand-in used here.'
    },
    {
        'version': 'v2_few_shot',
        'task': 'Classify sentiment after considering negation and evaluative words.',
        'allowed_labels': 'positive, negative',
        'notes': 'Adds simple examples in the verbal codebook.'
    }
])

prompt_log

In [ ]:
positive_words = {'good', 'great', 'excellent', 'wonderful', 'best', 'love', 'loved', 'happy', 'thanks', 'awesome', 'amazing'}
negative_words = {'bad', 'terrible', 'awful', 'worst', 'boring', 'hate', 'hated', 'sad', 'angry', 'poor', 'mess'}
negations = {'not', 'never', 'no'}


def offline_annotation(text, version='v1_zero_shot'):
    tokens = spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=1)
    score = 0
    for i, token in enumerate(tokens):
        value = int(token in positive_words) - int(token in negative_words)
        if version == 'v2_few_shot' and i > 0 and tokens[i - 1] in negations:
            value = -value
        score += value
    return 'positive' if score >= 0 else 'negative'

sample['llm_v1_label'] = sample['text'].apply(lambda x: offline_annotation(x, version='v1_zero_shot'))
sample['llm_v2_label'] = sample['text'].apply(lambda x: offline_annotation(x, version='v2_few_shot'))
sample[['text', 'polarity', 'llm_v1_label', 'llm_v2_label']].head()

In [ ]:
def metrics(y_true, y_pred):
    return pd.Series({
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, pos_label='positive'),
        'recall': recall_score(y_true, y_pred, pos_label='positive'),
        'f1': f1_score(y_true, y_pred, pos_label='positive')
    })

pd.DataFrame({
    'zero_shot_stand_in': metrics(sample['polarity'], sample['llm_v1_label']),
    'few_shot_stand_in': metrics(sample['polarity'], sample['llm_v2_label'])
}).T.round(3)

## 2a. Annotation confusion matrices

LLM-style annotation needs the same diagnostic discipline as supervised classification: which mistakes are being made, and are they substantively acceptable?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

ConfusionMatrixDisplay.from_predictions(
    sample['polarity'], sample['llm_v1_label'], ax=axes[0], cmap='Blues', colorbar=False
)
axes[0].set_title('Zero-shot stand-in')

ConfusionMatrixDisplay.from_predictions(
    sample['polarity'], sample['llm_v2_label'], ax=axes[1], cmap='Blues', colorbar=False
)
axes[1].set_title('Few-shot stand-in')

plt.tight_layout()

In [ ]:
disagreements = sample[sample['polarity'] != sample['llm_v2_label']]
disagreements[['text', 'polarity', 'llm_v2_label']].sample(10, random_state=3)

Optional live model idea: replace `offline_annotation()` with an Ollama or OpenAI call, but keep the same output column and validation code. Do not skip validation because the model is newer or larger.

In [ ]:
# Optional sketch only. Leave commented unless your environment is configured.
# from openai import OpenAI
# client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
# def live_annotation(text):
#     response = client.chat.completions.create(
#         model='your-local-model',
#         messages=[
#             {'role': 'system', 'content': 'Classify the post as positive or negative. Output only the label.'},
#             {'role': 'user', 'content': text}
#         ],
#         temperature=0
#     )
#     return response.choices[0].message.content.strip().lower()

## 3. Lightweight local document embeddings

For a classroom-friendly embedding workflow, we use TF-IDF followed by SVD to make dense vectors. This is not a transformer embedding, but it lets us practice the same operations: similarity, projection, and classifier training.

### Methodology formulas: embeddings, annotation, and semantic axes

An annotation model can be written as a function from text and a codebook or prompt $p$ to a label:

$$\hat{y}_i = f_p(x_i).$$

The local embedding section approximates dense document vectors by reducing a TF-IDF matrix:

$$Z = X V_k, \qquad z_i \in \mathbb{R}^k.$$

Documents are compared by cosine similarity in that embedding space:

$$\mathrm{sim}(i,j)=\frac{z_i^\top z_j}{\lVert z_i\rVert_2\lVert z_j\rVert_2}.$$

A semantic axis contrasts two anchor sets, for example positive versus negative texts:

$$a = \frac{\bar{z}_{+} - \bar{z}_{-}}{\lVert \bar{z}_{+} - \bar{z}_{-}\rVert_2}, \qquad s_i = z_i^\top a.$$

Classification on embeddings then estimates

$$P(y_i=1 \mid z_i)=\sigma(\gamma_0 + z_i^\top\gamma).$$

This is why prompt design, label validation, and embedding diagnostics belong together: all three define the measurement function that turns text into data.


In [ ]:
vectorizer = TfidfVectorizer(analyzer=spacy_analyzer, min_df=2, max_features=3000)
X = vectorizer.fit_transform(sample['text'])

svd = TruncatedSVD(n_components=50, random_state=42)
embeddings = svd.fit_transform(X)

embeddings.shape

## 3a. Embedding map

The first two SVD dimensions give a quick view of how posts are arranged in the dense vector space.

In [ ]:
sample['embedding_x'] = embeddings[:, 0]
sample['embedding_y'] = embeddings[:, 1]

plt.figure(figsize=(7, 5))
sns.scatterplot(data=sample, x='embedding_x', y='embedding_y', hue='polarity', alpha=0.75)
plt.title('Dense-vector map of TweetEval posts')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.tight_layout()

In [ ]:
def nearest_posts(row_index, n=6):
    sims = cosine_similarity(embeddings[row_index:row_index + 1], embeddings).ravel()
    order = sims.argsort()[::-1][1:n + 1]
    return sample.loc[order, ['text', 'polarity']].assign(similarity=sims[order])

print(sample.loc[0, 'text'])
nearest_posts(0, n=6)

## 4. Classification with dense vectors

Embeddings can be used as features in a supervised classifier. Compare this to yesterday's sparse TF-IDF classifier.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    sample['polarity'],
    test_size=0.25,
    random_state=42,
    stratify=sample['polarity']
)

embedding_model = LogisticRegression(max_iter=1000)
embedding_model.fit(X_train, y_train)
embedding_pred = embedding_model.predict(X_test)

metrics(y_test, embedding_pred).round(3)

## 4a. Classifier error map

Projecting predictions back into the embedding space helps identify regions where the classifier is uncertain or systematically wrong.

In [ ]:
embedding_eval = sample.copy()
embedding_eval['embedding_prediction'] = embedding_model.predict(embeddings)
embedding_eval['correct'] = embedding_eval['embedding_prediction'].eq(embedding_eval['polarity'])

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=embedding_eval,
    x='embedding_x',
    y='embedding_y',
    hue='polarity',
    style='correct',
    alpha=0.8
)
plt.title('Embedding classifier errors in vector space')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.tight_layout()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, embedding_pred, cmap='Blues')
plt.title('Classifier using dense document vectors')
plt.tight_layout()

## 5. Semantic axis

A semantic axis turns seed words into a direction in vector space. The measurement question is whether the chosen seeds validly represent the construct.

In [ ]:
def embed_texts(texts):
    return svd.transform(vectorizer.transform(texts))

positive_seed_words = ['great', 'love', 'happy', 'thanks', 'awesome']
negative_seed_words = ['bad', 'hate', 'sad', 'angry', 'worst']

pos_center = embed_texts(positive_seed_words).mean(axis=0)
neg_center = embed_texts(negative_seed_words).mean(axis=0)
axis = pos_center - neg_center
axis = axis / np.linalg.norm(axis)

sample['semantic_axis_score'] = embeddings @ axis
sample.groupby('polarity')['semantic_axis_score'].describe()

In [ ]:
sample.boxplot(column='semantic_axis_score', by='polarity', figsize=(6, 4))
plt.title('Semantic-axis scores by human label')
plt.suptitle('')
plt.ylabel('Score on positive-negative axis')
plt.tight_layout()

## 5a. Semantic-axis distribution

The axis is useful only if it separates cases in an interpretable and stable way. A distribution plot makes overlap visible.

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(data=sample, x='semantic_axis_score', hue='polarity', bins=30, element='step', stat='density', common_norm=False)
plt.axvline(0, color='black', linewidth=1)
plt.title('Distribution of semantic-axis scores')
plt.xlabel('Positive-negative semantic axis')
plt.tight_layout()

### Additional demo: semantic-axis sensitivity

Semantic axes depend on anchor words. This demonstration compares several plausible positive-negative axes and checks whether the class separation is stable.

In [ ]:
axis_specs = [
    {
        'axis_name': 'original seeds',
        'positive': ['great', 'love', 'happy', 'thanks', 'awesome'],
        'negative': ['bad', 'hate', 'sad', 'angry', 'worst']
    },
    {
        'axis_name': 'general sentiment',
        'positive': ['good', 'best', 'amazing', 'excellent', 'fun'],
        'negative': ['terrible', 'awful', 'poor', 'mess', 'wrong']
    },
    {
        'axis_name': 'social reaction',
        'positive': ['congrats', 'enjoy', 'smile', 'win', 'thank'],
        'negative': ['cry', 'fear', 'pain', 'fail', 'angry']
    }
]

axis_rows = []
for spec in axis_specs:
    pos = embed_texts(spec['positive']).mean(axis=0)
    neg = embed_texts(spec['negative']).mean(axis=0)
    candidate_axis = pos - neg
    norm = np.linalg.norm(candidate_axis)
    if norm == 0:
        continue
    candidate_axis = candidate_axis / norm
    scores = embeddings @ candidate_axis
    means = sample.assign(axis_score=scores).groupby('polarity')['axis_score'].mean()
    axis_rows.append({
        'axis_name': spec['axis_name'],
        'negative_mean': means.get('negative', np.nan),
        'positive_mean': means.get('positive', np.nan),
        'class_separation': means.get('positive', np.nan) - means.get('negative', np.nan)
    })

axis_sensitivity = pd.DataFrame(axis_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=axis_sensitivity, x='class_separation', y='axis_name', color='#4C72B0', ax=axes[0])
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('Positive-negative separation by axis')
axes[0].set_xlabel('Mean positive score - mean negative score')
axes[0].set_ylabel('')

extremes = pd.concat([
    sample.nsmallest(5, 'semantic_axis_score').assign(position='lowest on original axis'),
    sample.nlargest(5, 'semantic_axis_score').assign(position='highest on original axis')
])
sns.stripplot(data=extremes, x='semantic_axis_score', y='position', hue='polarity', jitter=0.15, ax=axes[1])
axes[1].set_title('Extreme documents on the original axis')
axes[1].set_xlabel('Semantic-axis score')
axes[1].set_ylabel('')

plt.tight_layout()
display(axis_sensitivity.round(3))
extremes[['position', 'polarity', 'semantic_axis_score', 'text']]

## 5b. Advanced extension: semantic use over time

This is a lightweight local analogue of semantic-drift analysis. We extract State of the Union sentences containing `freedom`, embed them with spaCy-tokenized TF-IDF/SVD, and visualize their position over time.

In [ ]:
sotu = pd.read_csv(DATA_DIR / 'sotu.csv')
sotu['year_numeric'] = pd.to_numeric(sotu['year'], errors='coerce')

rows = []
for _, row in sotu.dropna(subset=['text', 'year_numeric']).iterrows():
    if row['year_numeric'] < 1900 or row['party'] not in ['Democratic', 'Republican']:
        continue
    doc = nlp(str(row['text']))
    for sent in doc.sents:
        sent_text = sent.text.strip()
        if re.search(r'\bfreedom\b', sent_text, flags=re.IGNORECASE):
            rows.append({
                'year': int(row['year_numeric']),
                'decade': int(row['year_numeric'] // 10 * 10),
                'party': row['party'],
                'sentence': sent_text
            })

freedom = pd.DataFrame(rows)
freedom.head(), freedom.shape

In [ ]:
if len(freedom) >= 20:
    freedom_vec = TfidfVectorizer(analyzer=spacy_analyzer, min_df=2, max_features=1500)
    freedom_X = freedom_vec.fit_transform(freedom['sentence'])
    freedom_svd = TruncatedSVD(n_components=2, random_state=42)
    freedom_coords = freedom_svd.fit_transform(freedom_X)
    freedom['x'] = freedom_coords[:, 0]
    freedom['y'] = freedom_coords[:, 1]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.scatterplot(data=freedom, x='x', y='y', hue='party', size='year', alpha=0.75, ax=axes[0])
    axes[0].set_title("Sentences containing 'freedom': semantic map")
    axes[0].set_xlabel('Component 1')
    axes[0].set_ylabel('Component 2')

    freq = freedom.groupby(['decade', 'party']).size().reset_index(name='sentences')
    sns.lineplot(data=freq, x='decade', y='sentences', hue='party', marker='o', ax=axes[1])
    axes[1].set_title("Frequency of 'freedom' sentences by decade")
    axes[1].set_xlabel('Decade')
    axes[1].set_ylabel('Number of sentences')

    plt.tight_layout()
else:
    print('Not enough freedom sentences for the visualization.')

## 6. Student task

Change the seed words and check whether the axis is stable. Then answer:

1. What concept do the seed words define?
2. Which examples score unexpectedly high or low?
3. What validation evidence would you need before publication?

In [ ]:
# Your turn: replace these seed words and recompute the axis.
student_positive_seeds = ['good', 'great', 'best']
student_negative_seeds = ['bad', 'worst', 'poor']

# Add your sensitivity check here.